## 03_train_random_forest  (OLD workspace) — Practice model artifact 3 of 5

Trains on the table chosen in 00_setup_sample_data (practice_customers):
  features: tenure_months, churn_risk, seg_enterprise, seg_retail
  label:    monthly_revenue


In [ ]:

%run ./lib/config_lib


In [ ]:
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (OLD schema name — unchanged by migration)")

CFG = setup_config(
    config_path="",
    run_date="all",
    schema=dbutils.widgets.get("SCHEMA").strip(),
)

TRAIN_TABLE = CFG.table("practice_customers")
print(f"Training table: {TRAIN_TABLE}")
print(f"features: {FEATURES} | label: {LABEL}")


In [ ]:

import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X, y, _ids = load_training_data(TRAIN_TABLE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"rows: {len(y)} | train: {len(y_train)} | test: {len(y_test)}")

model = RandomForestRegressor(n_estimators=40, max_depth=8, random_state=42).fit(X_train, y_train)
rmse = float(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))
print(f"RMSE: {rmse:.4f}")


In [ ]:

import mlflow

REG_NAME = CFG.model("practice_random_forest")  # Free Edition UC registry: <current_catalog>.default.<name>
with mlflow.start_run() as run:
    mlflow.log_param("model_type", "random_forest")
    mlflow.log_param("n_estimators", 40)
    mlflow.log_param("train_table", TRAIN_TABLE)
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(model, artifact_path="model")
    mv = mlflow.register_model(f"runs:/{run.info.run_id}/model", REG_NAME)

print(f"REGISTERED  name='{mv.name}'  version={mv.version}")
